# Card churn: starter notebook

**Task.** For each customer in `test.csv`, predict the probability that the customer closes the credit card within 3 months after the snapshot date (30 June 2026). The target column is `churn_3m` (1 = closed).

**Metric.** ROC AUC: it only checks the *ranking* (are churners given higher scores than non-churners?). 0.5 = random, 1.0 = perfect.

This notebook builds a simple baseline: numbers only, no cleaning, logistic regression. The data is messy on purpose; improving on this baseline is your job.

**Setup.** Import the libraries and find the data folder (the same code works on Kaggle and on your laptop).

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

import os, glob
# works on Kaggle (/kaggle/input/...) and locally (files next to the notebook)
hits = (glob.glob("/kaggle/input/**/train.csv", recursive=True)   # on Kaggle
        + glob.glob("train.csv")                                  # next to the notebook
        + glob.glob("data/train.csv"))                            # in a data/ folder
DATA = (os.path.dirname(hits[0]) or ".") if hits else "."
print("data folder:", DATA)

data folder: .


**Load the data.** `train` has the answer column `churn_3m`; `test` does not.

In [2]:
train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
print(train.shape, test.shape)               # (rows, columns)
train.head()                                 # first 5 rows

(12120, 24) (8000, 23)


,customer_id,bank_code,age,gender,province,occupation,income_month,card_type,open_date,credit_limit,...,spend_change_pct,app_logins_30d,complaints_12m,late_payments_12m,salary_via_bank,other_bank_cards,promo_ended_60d,last_contact_channel,currency,churn_3m
0,CUSTI3FNUD,VB01,48.0,F,Hải Dương,self_employed,25.1,Gold,2026-04-09,55.0,...,-23.5,2.0,0,0,N,1.0,0,app,vnd,1
1,CUSOOYRASU,VB01,47.0,M,Hà Nội,Salaried,NaN,Classic,06/01/2024,30.0,...,-27.7,4.0,0,0,Y,1.0,0,app,VND,0
2,CUSQTUBJRJ,VB01,38.0,M,Long An,other,8.5,Cashback,02/08/2022,15.0,...,-20.7,8.0,0,0,N,0.0,1,none,VND,0
3,CUSD7I53QY,VB01,25.0,M,TP.HCM,self_employed,33.9,Cashback,12/05/2022,110.0,...,44.7,5.0,0,0,N,NaN,1,app,VND,0
4,CUS02NC6FN,VB01,32.0,m,TP.HCM,civil_servant,14.0,Cashback,03/07/2025,40.0,...,-28.0,4.0,0,0,N,0.0,0,none,VND,0


## A first look
**How rare is churn, and which columns have missing values?** A low churn rate means accuracy is a poor metric: predicting "no churn" for everyone is already right most of the time.

In [3]:
print("churn rate:", round(train.churn_3m.mean(), 3))   # share of 1s
# share of missing values per column, largest first
train.isna().mean().sort_values(ascending=False).round(3).head(10)

churn rate: 0.117


other_bank_cards        0.203
income_month            0.129
app_logins_30d          0.125
salary_via_bank         0.059
occupation              0.051
last_contact_channel    0.042
age                     0.033
gender                  0.019
province                0.017
open_date               0.006
dtype: float64

**Text columns.** Count the values of a few text columns. Are there several ways of writing the same thing?

In [4]:
# categorical columns deserve a closer look
for col in ["gender", "province", "occupation"]:
    print(train[col].value_counts(dropna=False).head(8), "\n")   # dropna=False also counts NaN

gender
F         5572
M         4778
female     280
male       278
F          264
Nữ         255
Nam        235
NaN        231
Name: count, dtype: int64 

province
TP.HCM        2598
Hà Nội        2587
Hải Phòng      639
Đà Nẵng        506
Bình Dương     424
Cần Thơ        408
Đồng Nai       293
TP HCM         247
Name: count, dtype: int64 

occupation
salaried         4680
self_employed    2098
civil_servant    1079
other             998
retired           718
NaN               614
student           281
SALARIED          267
Name: count, dtype: int64 



**Number columns.** `describe()` gives min, max, mean and quartiles. Check that every min and max makes sense for that column.

In [5]:
train.describe().T.round(2)                  # .T: one row per column

,count,mean,std,min,25%,50%,75%,max
age,11724.0,39.74,29.94,0.00,31.00,38.00,45.00,9.990000e+02
income_month,10558.0,187584.30,2065906.52,1.00,11.80,17.70,26.10,5.050000e+07
credit_limit,12120.0,1094113.39,10527161.79,10.00,25.00,40.00,70.00,3.850000e+08
annual_fee_waived,12120.0,0.51,0.50,0.00,0.00,1.00,1.00,1.000000e+00
utilization_6m,12120.0,0.32,0.72,-1.00,0.14,0.27,0.42,1.500000e+01
txn_count_3m,12120.0,18.91,10.27,1.00,12.00,17.00,24.00,1.010000e+02
txn_amount_3m,12120.0,25.34,28.81,-2.99,8.72,16.76,31.63,4.628100e+02
spend_change_pct,12120.0,-0.10,24.85,-95.00,-16.90,-0.00,16.60,1.058000e+02
app_logins_30d,10604.0,6.37,4.63,0.00,3.00,5.00,8.00,4.800000e+01
complaints_12m,12120.0,0.25,0.50,0.00,0.00,0.00,0.00,3.000000e+00


## Baseline: numeric columns only, logistic regression
Keep only the numeric columns, fill missing values with the median of `train`, scale them, and fit a logistic regression. We measure it with **5-fold cross-validation**: the training data is cut into 5 parts, and each part is used once for scoring a model trained on the other 4. This estimate is more reliable than the public leaderboard.

In [6]:
num = test.select_dtypes("number").columns.tolist()   # numeric columns only
fill = train[num].median()                   # one median per column, from train
X, y = train[num].fillna(fill), train.churn_3m
# StandardScaler: mean 0, std 1 per column, so that no column dominates
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
# StratifiedKFold keeps the same churn rate in each of the 5 parts
cv = StratifiedKFold(5, shuffle=True, random_state=0)
auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
print("5-fold AUC:", auc.round(3), "mean", auc.mean().round(3))

5-fold AUC: [0.707 0.719 0.736 0.75  0.733] mean 0.729


**Submission.** Fit the model on all of `train`, predict a probability for each test customer, and write `submission.csv` with the two columns Kaggle expects: `customer_id`, `churn_3m`.

In [7]:
model.fit(X, y)                              # learn from all training rows
sub = pd.DataFrame({"customer_id": test.customer_id,
                    # predict_proba gives [P(0), P(1)]; keep P(churn = 1)
                    "churn_3m": model.predict_proba(test[num].fillna(fill))[:, 1]})
sub.to_csv("submission.csv", index=False)
sub.head()

,customer_id,churn_3m
0,CUSJCOH081,0.038814
1,CUSFDBKK99,0.138019
2,CUS7DPD79Y,0.047280
3,CUSDCOU57Y,0.045222
4,CUS4NO88FF,0.028404


## Where to go next

- Look at every column carefully: what could be wrong in it?
- Decide what a missing value *means* before you fill it.
- Use the text and date columns, not only the numbers.
- Try models that capture non-linear effects and interactions.
- Always validate with cross-validation, not with the public leaderboard.